# EDA Phase 2 — Symptoms and Lifestyle Analysis

**Dataset:** `datasets/cleaned_dataset.csv`  
**Scope:** Symptom prevalence, burden, co-occurrence, and lifestyle context.  
**Constraint:** No machine learning / predictive modeling.

This notebook is the day-level (record-level) follow-up to Phase 1 user/cycle structure. Grain is **one row = one study day** for a participant (`id`).

## Setup

Ordinal symptom fields use labels such as `Not at all`, `Very Low/Little`, `Low`, `Moderate`, `High`, `Very High`.  
**Presence rule (documented):** a categorical symptom is *present* if the value is not a “not at all” variant (case-insensitive).  
`pain_nrs` is numeric (NRS-like); **notable pain** is defined as `pain_nrs >= 4` (moderate or higher).  
`mood` is a state label; **mood disturbance** is `Moody` or `sad` (vs `Calm` / `Energetic`).

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("..").resolve()
if not (ROOT / "datasets" / "cleaned_dataset.csv").exists():
    ROOT = Path.cwd()
    if not (ROOT / "datasets" / "cleaned_dataset.csv").exists():
        ROOT = Path.cwd().parent

DATA_PATH = ROOT / "datasets" / "cleaned_dataset.csv"
FIG_DIR = ROOT / "reports" / "figures"
REPORT_PATH = ROOT / "reports" / "eda2_summary.md"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

SYMPTOM_COLS = [
    "headaches", "cramps", "sorebreasts", "fatigue", "sleepissue",
    "moodswing", "stress", "foodcravings", "indigestion", "bloating",
    "appetite", "pain_nrs", "pain_location", "mood",
]
ORDINAL_SYMPTOMS = [
    "headaches", "cramps", "sorebreasts", "fatigue", "sleepissue",
    "moodswing", "stress", "foodcravings", "indigestion", "bloating", "appetite",
]
LIFESTYLE_COLS = [
    "exerciselevel", "activity_level", "sleep_hours", "BMI", "age", "contraception_type",
]
ABSENT_TOKENS = {"not at all", "notatall"}
MOOD_DISTURBANCE = {"moody", "sad"}
PAIN_NRS_THRESHOLD = 4.0


def save_fig(fig, name: str) -> Path:
    path = FIG_DIR / name
    fig.savefig(path)
    plt.close(fig)
    print(f"saved {path.relative_to(ROOT)}")
    return path


def is_absent_label(val) -> bool:
    if pd.isna(val):
        return True
    token = str(val).strip().lower().replace("  ", " ")
    return token in ABSENT_TOKENS


def present_mask(series: pd.Series, col: str) -> pd.Series:
    """Binary presence for co-occurrence / burden."""
    if col == "pain_nrs":
        return pd.to_numeric(series, errors="coerce") >= PAIN_NRS_THRESHOLD
    if col == "mood":
        return series.astype(str).str.strip().str.lower().isin(MOOD_DISTURBANCE)
    if col == "pain_location":
        return ~series.map(is_absent_label)
    return ~series.map(is_absent_label)

print("ROOT", ROOT)
print("DATA", DATA_PATH)

## 1. Dataset Validation

Confirm every requested column exists, then report row counts, unique users, and missingness on the analysis set.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

required = SYMPTOM_COLS + LIFESTYLE_COLS + ["id"]
missing_cols = [c for c in required if c not in df.columns]
print(f"rows={len(df):,}  cols={df.shape[1]}  unique users={df['id'].nunique()}")
print("missing required columns:", missing_cols if missing_cols else "none")

analysis_cols = required
missing_tbl = (
    pd.DataFrame({
        "column": analysis_cols,
        "dtype": [str(df[c].dtype) if c in df.columns else "MISSING" for c in analysis_cols],
        "n_missing": [int(df[c].isna().sum()) if c in df.columns else np.nan for c in analysis_cols],
        "pct_missing": [
            round(100 * df[c].isna().mean(), 3) if c in df.columns else np.nan
            for c in analysis_cols
        ],
        "n_unique": [int(df[c].nunique(dropna=True)) if c in df.columns else np.nan for c in analysis_cols],
    })
    .set_index("column")
)
display(missing_tbl)

assert not missing_cols, f"Missing columns: {missing_cols}"

**Validation notes**

- Grain is day-level (`id` + `day_in_study`); do not treat row counts as unique people.
- Expected sparse missingness on `sorebreasts` and `mood` only among the requested fields.
- Watch data-quality oddities later: `pain_nrs` max can exceed a 0–10 NRS, `mood` may contain a leaked `pain_location` token, and `pain_location` has typos.

## 2. Symptom Prevalence

Prevalence = percent of **records** where the symptom is present under the rules in Setup. Ranked bar chart + top 5.

In [ ]:
presence = pd.DataFrame({c: present_mask(df[c], c) for c in SYMPTOM_COLS})
prevalence = (100 * presence.mean()).sort_values(ascending=False).rename("prevalence_pct")
n_present = presence.sum().rename("n_present")
prev_tbl = pd.concat([n_present, prevalence], axis=1).sort_values("prevalence_pct", ascending=False)
prev_tbl["n_present"] = prev_tbl["n_present"].astype(int)
display(prev_tbl.round(2))

top5 = prev_tbl.head(5)
print("Top 5 most common symptoms:")
print(top5.round(2))

fig, ax = plt.subplots(figsize=(10, 6))
order = prev_tbl.index.tolist()
sns.barplot(x=prev_tbl["prevalence_pct"], y=order, ax=ax, color="#2E86AB")
ax.set_xlabel("Prevalence (% of records)")
ax.set_ylabel("Symptom")
ax.set_title("Symptom Prevalence (day-level records)")
for i, (sym, row) in enumerate(prev_tbl.iterrows()):
    ax.text(row["prevalence_pct"] + 0.6, i, f"{row['prevalence_pct']:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, min(100, prev_tbl["prevalence_pct"].max() + 12))
fig_prev = save_fig(fig, "eda2_symptom_prevalence.png")

**How to read prevalence**

- `appetite` will be near-universal if “any non-zero appetite” counts as present; that is a scale design issue, not necessarily a clinical finding.
- `pain_nrs` prevalence depends on the `>= 4` cutoff; almost all rows have `pain_nrs >= 2`, so a `> 0` rule would be uninformative.
- Rankings are **record-weighted**. Heavy-logging users contribute more days.

## 3. Symptom Burden

Burden = count of present symptoms on each record (0–14). Histogram, mean, and median.

In [ ]:
df["symptom_count"] = presence.sum(axis=1).astype(int)
burden_mean = float(df["symptom_count"].mean())
burden_median = float(df["symptom_count"].median())
print(f"mean symptom count = {burden_mean:.2f}")
print(f"median symptom count = {burden_median:.1f}")
print(df["symptom_count"].describe())

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df["symptom_count"], bins=range(0, 16), discrete=True, ax=ax, color="#2E86AB")
ax.axvline(burden_mean, color="#E94F37", linestyle="--", label=f"mean={burden_mean:.2f}")
ax.axvline(burden_median, color="#F6AE2D", linestyle="-.", label=f"median={burden_median:.1f}")
ax.set_xlabel("Number of symptoms present on the record")
ax.set_ylabel("Number of records")
ax.set_title("Symptom Burden per Record")
ax.legend()
fig_burden = save_fig(fig, "eda2_symptom_burden_hist.png")

A right- or left-shifted burden histogram tells us whether days are typically oligosymptomatic or multi-symptom clusters — important before any later “anomaly” work (still not modeling here).

## 4. Symptom Co-occurrence

Binary co-occurrence: Pearson correlation of presence flags (scale-free association) plus raw joint counts for the strongest pairs.

In [ ]:
co_corr = presence.astype(float).corr()
joint = presence.astype(int).T.dot(presence.astype(int))
n = len(presence)
joint_pct = 100 * joint / n

pairs = []
cols = list(presence.columns)
for i, a in enumerate(cols):
    for b in cols[i + 1:]:
        pairs.append({
            "symptom_a": a,
            "symptom_b": b,
            "both_n": int(joint.loc[a, b]),
            "both_pct": float(joint_pct.loc[a, b]),
            "corr": float(co_corr.loc[a, b]),
        })
pairs_df = pd.DataFrame(pairs).sort_values("corr", ascending=False)
print("Strongest pairs by presence correlation:")
display(pairs_df.head(10).round(3))
print("Strongest pairs by joint prevalence (% of records with both):")
display(pairs_df.sort_values("both_pct", ascending=False).head(10).round(3))

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    co_corr,
    cmap="RdBu_r",
    center=0,
    vmin=-0.4,
    vmax=0.8,
    square=True,
    ax=ax,
    cbar_kws={"label": "Pearson r (binary presence)"},
)
ax.set_title("Symptom Co-occurrence Heatmap (presence correlation)")
fig_heat = save_fig(fig, "eda2_symptom_cooccurrence_heatmap.png")

High **joint %** can be driven by two common symptoms (e.g. appetite × fatigue). High **correlation** is more informative of clustering beyond base rates.

## 5. Lifestyle Analysis

Distributions and summary statistics for sleep, BMI, age, exercise, activity, and contraception. No outcomes modeling.

In [ ]:
numeric_life = ["sleep_hours", "BMI", "age"]
life_stats = df[numeric_life].describe().T
life_stats["median"] = df[numeric_life].median()
display(life_stats.round(3))

print("exerciselevel")
display(df["exerciselevel"].value_counts(dropna=False).to_frame("n"))
print("activity_level")
display(df["activity_level"].value_counts(dropna=False).to_frame("n"))
print("contraception_type")
display(df["contraception_type"].value_counts(dropna=False).to_frame("n"))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
sns.histplot(df["sleep_hours"], bins=20, kde=True, ax=axes[0, 0], color="#2E86AB")
axes[0, 0].set_title("Sleep hours")
axes[0, 0].set_xlabel("Hours")

sns.histplot(df["BMI"], bins=20, kde=True, ax=axes[0, 1], color="#2E86AB")
axes[0, 1].set_title("BMI")
axes[0, 1].set_xlabel("kg/m²")

sns.histplot(df["age"], bins=sorted(df["age"].unique()), discrete=True, ax=axes[0, 2], color="#2E86AB")
axes[0, 2].set_title("Age")
axes[0, 2].set_xlabel("Years")

ex_order = ["Not at all", "Very Low", "Low", "Moderate", "High", "Very High"]
ex_counts = df["exerciselevel"].value_counts().reindex(
    [x for x in ex_order if x in set(df["exerciselevel"].dropna())]
).fillna(0)
sns.barplot(x=ex_counts.values, y=ex_counts.index, ax=axes[1, 0], color="#A23B72")
axes[1, 0].set_title("Exercise level")
axes[1, 0].set_xlabel("Records")

act_counts = df["activity_level"].value_counts()
sns.barplot(x=act_counts.values, y=act_counts.index, ax=axes[1, 1], color="#A23B72")
axes[1, 1].set_title("Activity level")
axes[1, 1].set_xlabel("Records")

ct_counts = df["contraception_type"].value_counts()
sns.barplot(x=ct_counts.values, y=ct_counts.index, ax=axes[1, 2], color="#A23B72")
axes[1, 2].set_title("Contraception type")
axes[1, 2].set_xlabel("Records")

fig.suptitle("Lifestyle Distributions (record-level)", y=1.02)
fig.tight_layout()
fig_life = save_fig(fig, "eda2_lifestyle_distributions.png")

Age is young-adult and discrete (few unique values). BMI and sleep should be interpreted as **day-weighted** across 42 users, not a population survey.

## 6. User-Level Analysis

Average symptom count per user, top users by mean burden, and the between-user distribution.

In [ ]:
user_burden = (
    df.groupby("id")
    .agg(
        n_records=("symptom_count", "size"),
        mean_symptom_count=("symptom_count", "mean"),
        median_symptom_count=("symptom_count", "median"),
        max_symptom_count=("symptom_count", "max"),
    )
    .sort_values("mean_symptom_count", ascending=False)
)
print("Between-user mean of mean-burden:", user_burden["mean_symptom_count"].mean())
print("Median user mean-burden:", user_burden["mean_symptom_count"].median())
print("Min / max user mean-burden:",
      user_burden["mean_symptom_count"].min(), user_burden["mean_symptom_count"].max())
print("\nTop 10 users by mean symptom burden:")
display(user_burden.head(10).round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(user_burden["mean_symptom_count"], bins=12, ax=axes[0], color="#2E86AB")
axes[0].axvline(user_burden["mean_symptom_count"].mean(), color="#E94F37", linestyle="--",
                label=f"mean={user_burden['mean_symptom_count'].mean():.2f}")
axes[0].set_title("Distribution of user-level mean symptom burden")
axes[0].set_xlabel("Mean symptoms per day (user)")
axes[0].legend()

top10 = user_burden.head(10).reset_index()
top10["id"] = top10["id"].astype(str)
sns.barplot(data=top10, y="id", x="mean_symptom_count", ax=axes[1], color="#2E86AB")
axes[1].set_title("Top 10 users by mean symptom burden")
axes[1].set_xlabel("Mean symptoms per record")
axes[1].set_ylabel("User id")
fig.tight_layout()
fig_user = save_fig(fig, "eda2_user_symptom_burden.png")

If a few users dominate high burden, later personalization should not pool all days equally without a user effect (descriptive warning only).

## 7. Missingness Analysis

Missing counts and percentages for all symptom + lifestyle columns (plus `id`).

In [ ]:
miss = missing_tbl.reset_index().sort_values("pct_missing", ascending=False)
display(miss)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=miss, x="pct_missing", y="column", ax=ax, color="#E94F37")
ax.set_xlabel("Missing (%)")
ax.set_ylabel("Column")
ax.set_title("Missingness by Analysis Column")
fig_miss = save_fig(fig, "eda2_missingness.png")

# Extra quality flags (not missingness, but related)
print("pain_nrs > 10 (implausible NRS):", int((df["pain_nrs"] > 10).sum()))
print("mood unexpected labels:", sorted(set(df["mood"].dropna().astype(str)) - {"Moody", "Calm", "Energetic", "sad"}))

## 8. Insights and report export

Write `reports/eda2_summary.md` from the computed tables so the markdown report stays in sync with the notebook.

In [ ]:
top5_lines = "\n".join(
    f"| {i} | `{idx}` | {int(r.n_present)} | {r.prevalence_pct:.2f}% |"
    for i, (idx, r) in enumerate(top5.iterrows(), start=1)
)
prev_lines = "\n".join(
    f"| `{idx}` | {int(r.n_present)} | {r.prevalence_pct:.2f}% |"
    for idx, r in prev_tbl.iterrows()
)
strong_corr = pairs_df.head(5)
corr_lines = "\n".join(
    f"| `{r.symptom_a}` | `{r.symptom_b}` | {r.corr:.3f} | {r.both_pct:.2f}% | {r.both_n} |"
    for r in strong_corr.itertuples()
)
strong_joint = pairs_df.sort_values("both_pct", ascending=False).head(5)
joint_lines = "\n".join(
    f"| `{r.symptom_a}` | `{r.symptom_b}` | {r.both_pct:.2f}% | {r.corr:.3f} |"
    for r in strong_joint.itertuples()
)
ex_vc = df["exerciselevel"].value_counts()
act_vc = df["activity_level"].value_counts()
ct_vc = df["contraception_type"].value_counts()
ex_lines = "\n".join(f"| `{k}` | {v} | {100*v/len(df):.2f}% |" for k, v in ex_vc.items())
act_lines = "\n".join(f"| `{k}` | {v} | {100*v/len(df):.2f}% |" for k, v in act_vc.items())
ct_lines = "\n".join(f"| `{k}` | {v} | {100*v/len(df):.2f}% |" for k, v in ct_vc.items())
miss_nz = miss[miss["n_missing"] > 0]
if len(miss_nz):
    miss_lines = "\n".join(
        f"| `{r.column}` | {int(r.n_missing)} | {r.pct_missing:.3f}% |"
        for r in miss_nz.itertuples()
    )
else:
    miss_lines = "| — | 0 | 0% |"

top_users = user_burden.head(10)
user_lines = "\n".join(
    f"| `{idx}` | {int(r.n_records)} | {r.mean_symptom_count:.2f} | {r.median_symptom_count:.1f} | {int(r.max_symptom_count)} |"
    for idx, r in top_users.iterrows()
)

sleep = df["sleep_hours"]
bmi = df["BMI"]
age = df["age"]

md = f"""# EDA Phase 2 — Symptoms and Lifestyle

**Dataset:** `cleaned_dataset.csv`  
**Rows:** {len(df):,} day-level records · **Users:** {df['id'].nunique()}  
**Notebook:** `notebooks/eda_02_symptom_lifestyle.ipynb`  
**Scope:** Descriptive only (no predictive models)

### Presence definitions
- Ordinal symptoms (`headaches` … `appetite`): present if label is not a “Not at all” variant.
- `pain_location`: present if not a “not at all” variant (typos still count as a location).
- `pain_nrs`: notable if ≥ {PAIN_NRS_THRESHOLD:.0f} (moderate+). Raw NRS is almost never zero in this file.
- `mood`: disturbance if `Moody` or `sad`.

---

## 1. Dataset validation

All requested symptom and lifestyle columns are present.

Columns with any missing values among the analysis set:

| column | n missing | % |
|--------|-----------|---|
{miss_lines}

Quality flags (not missingness): `pain_nrs > 10` on **{int((df['pain_nrs'] > 10).sum())}** rows; `mood` contains unexpected labels: `{sorted(set(df['mood'].dropna().astype(str)) - set(['Moody','Calm','Energetic','sad']))}`.

---

## 2. Symptom prevalence

| symptom | n present | prevalence |
|---------|-----------|------------|
{prev_lines}

### Top 5 most common

| rank | symptom | n | % |
|------|---------|---|---|
{top5_lines}

![Symptom prevalence](figures/eda2_symptom_prevalence.png)

---

## 3. Symptom burden

| metric | value |
|--------|-------|
| Mean symptoms / record | {burden_mean:.2f} |
| Median symptoms / record | {burden_median:.1f} |
| Min / max | {int(df['symptom_count'].min())} / {int(df['symptom_count'].max())} |

![Burden histogram](figures/eda2_symptom_burden_hist.png)

---

## 4. Co-occurrence

Strongest pairs by **binary correlation**:

| a | b | r | joint % | joint n |
|---|---|---|---------|--------|
{corr_lines}

Strongest pairs by **joint prevalence**:

| a | b | joint % | r |
|---|---|---------|---|
{joint_lines}

![Co-occurrence heatmap](figures/eda2_symptom_cooccurrence_heatmap.png)

---

## 5. Lifestyle

| variable | mean | median | std | min | max |
|----------|------|--------|-----|-----|-----|
| sleep_hours | {sleep.mean():.2f} | {sleep.median():.2f} | {sleep.std():.2f} | {sleep.min():.2f} | {sleep.max():.2f} |
| BMI | {bmi.mean():.2f} | {bmi.median():.2f} | {bmi.std():.2f} | {bmi.min():.2f} | {bmi.max():.2f} |
| age | {age.mean():.2f} | {age.median():.1f} | {age.std():.2f} | {age.min():.0f} | {age.max():.0f} |

### exerciselevel
| level | n | % |
|-------|---|---|
{ex_lines}

### activity_level
| level | n | % |
|-------|---|---|
{act_lines}

### contraception_type
| type | n | % |
|------|---|---|
{ct_lines}

![Lifestyle distributions](figures/eda2_lifestyle_distributions.png)

---

## 6. User-level burden

| metric | value |
|--------|-------|
| Mean of user-mean burden | {user_burden['mean_symptom_count'].mean():.2f} |
| Median user-mean burden | {user_burden['mean_symptom_count'].median():.2f} |
| Min / max user-mean | {user_burden['mean_symptom_count'].min():.2f} / {user_burden['mean_symptom_count'].max():.2f} |

### Top 10 users by mean symptom count

| id | records | mean | median | max |
|----|---------|------|--------|-----|
{user_lines}

![User burden](figures/eda2_user_symptom_burden.png)

---

## 7. Missingness

![Missingness](figures/eda2_missingness.png)

Requested fields are nearly complete. Missingness is not the main quality issue; label noise (`pain_location` typos, `mood` leakage, `pain_nrs` > 10) is.

---

## 8. Findings

1. **Systemic/energy symptoms dominate ordinal “any intensity” prevalence** (fatigue, stress, sleep issues, cravings) more than classic pelvic symptoms such as cramps or breast tenderness, which are often logged as “Not at all”.
2. **`appetite` is almost always non-zero**, so ranking it as a “symptom” inflates prevalence; treat it as a lifestyle/appetite-state field in later phases.
3. **Typical day is multi-symptom**: mean burden ≈ {burden_mean:.1f} of 14 flags (median {burden_median:.0f}). Isolated single-symptom days are not the norm under these encodings.
4. **Co-occurrence vs joint %:** pairs involving two high-base-rate symptoms look common jointly; correlation is the better cue for clustering (e.g. mood-adjacent or GI clusters if they appear in the table above).
5. **Lifestyle cohort is young** (ages {int(age.min())}–{int(age.max())}), sleep centers near {sleep.median():.1f} h, BMI mean {bmi.mean():.1f}. Exercise is mostly Low/Moderate; activity_level is only Low vs Moderate. Most records are `contraception_type = Not at all`.
6. **Users differ in burden** (user-mean range {user_burden['mean_symptom_count'].min():.1f}–{user_burden['mean_symptom_count'].max():.1f}). Pooling days without a user lens will mix high-burden and low-burden phenotypes.
7. **Clean next steps (still not ML):** recode `pain_location` typos, cap or audit `pain_nrs` > 10, drop leaked `mood` values, and consider intensity-weighted scores instead of binary presence.

*Generated by `notebooks/eda_02_symptom_lifestyle.ipynb`*
"""

REPORT_PATH.write_text(md, encoding="utf-8")
print("wrote", REPORT_PATH.relative_to(ROOT))